## 01 — Writing the LOD Files

We have a design. Now we execute it.

This notebook:

1. Defines the four LOD configurations
2. Implements the simplification pipeline
3. Writes four GeoJSON files to `data/lod/`
4. Reports what was produced

Run this notebook once. The output files are used by every module that follows.

## Setup

In [ ]:
import json
import time
from pathlib import Path
from shapely.geometry import LineString

data_path  = Path("../../data/ne_10m_railroads.geojson")
output_dir = Path("../../data/lod")
output_dir.mkdir(parents=True, exist_ok=True)

with open(data_path) as f:
    railroads = json.load(f)

features = railroads["features"]
print(f"Loaded {len(features):,} features")
print(f"Output directory: {output_dir.resolve()}")

## LOD Configuration

Each level is defined by three parameters:
- `epsilon` — the simplification tolerance in degrees
- `max_scalerank` — maximum scalerank to include (`None` = include all)
- `zoom_range` — informational, used in the output properties

In [ ]:
LOD_LEVELS = [
    {
        "name":          "coarse",
        "filename":      "railroads_coarse.geojson",
        "epsilon":       1.0,
        "max_scalerank": 4,
        "zoom_range":    "1-3",
    },
    {
        "name":          "medium",
        "filename":      "railroads_medium.geojson",
        "epsilon":       0.1,
        "max_scalerank": None,
        "zoom_range":    "4-6",
    },
    {
        "name":          "fine",
        "filename":      "railroads_fine.geojson",
        "epsilon":       0.01,
        "max_scalerank": None,
        "zoom_range":    "7-10",
    },
    {
        "name":          "extra_fine",
        "filename":      "railroads_extra_fine.geojson",
        "epsilon":       0.001,
        "max_scalerank": None,
        "zoom_range":    "11+",
    },
]

## The Simplification Function

One function handles a single feature at a given epsilon. It returns the simplified feature dict, or `None` if the feature collapses.

In [2]:
def simplify_feature(feature, epsilon):
    """
    Simplify one GeoJSON LineString feature using Shapely's Douglas-Peucker.
    Returns a new feature dict, or None if the result has fewer than 2 points.
    """
    coords = feature["geometry"]["coordinates"]

    simplified_geom = LineString(coords).simplify(epsilon, preserve_topology=False)
    simplified_coords = list(simplified_geom.coords)

    if len(simplified_coords) < 2:
        return None

    return {
        "type":       "Feature",
        "properties": feature["properties"],
        "geometry":   {
            "type":        "LineString",
            "coordinates": simplified_coords,
        },
    }

## The Pipeline

For each LOD level:
1. Filter by `max_scalerank` if set
2. Simplify each feature
3. Drop `None` results (collapsed features)
4. Write to a GeoJSON file

In [3]:
print(f"{'Level':<12} {'Epsilon':>8} {'In':>8} {'Out':>8} {'Dropped':>9} {'Size (MB)':>11} {'Time (s)':>10}")
print("-" * 72)

for level in LOD_LEVELS:
    t0 = time.perf_counter()

    # Step 1 — feature filter
    if level["max_scalerank"] is not None:
        candidates = [
            f for f in features
            if f["properties"]["scalerank"] <= level["max_scalerank"]
        ]
    else:
        candidates = features

    # Step 2 & 3 — simplify and drop collapsed
    simplified = []
    for f in candidates:
        result = simplify_feature(f, level["epsilon"])
        if result is not None:
            simplified.append(result)

    # Step 4 — write output
    collection = {"type": "FeatureCollection", "features": simplified}
    out_path = output_dir / level["filename"]

    with open(out_path, "w") as f:
        json.dump(collection, f)

    elapsed   = time.perf_counter() - t0
    size_mb   = out_path.stat().st_size / 1_000_000
    dropped   = len(candidates) - len(simplified)

    print(
        f"{level['name']:<12} {level['epsilon']:>8} "
        f"{len(candidates):>8,} {len(simplified):>8,} "
        f"{dropped:>9,} {size_mb:>10.2f} {elapsed:>10.2f}"
    )

print("-" * 72)
print(f"\nAll files written to: {output_dir.resolve()}")

Level         Epsilon       In      Out   Dropped   Size (MB)   Time (s)
------------------------------------------------------------------------


NameError: name 'LOD_LEVELS' is not defined

## Verifying the Output Files

Let's confirm the files exist and are valid GeoJSON by loading one back.

In [ ]:
print("Output files:")
for level in LOD_LEVELS:
    path = output_dir / level["filename"]
    size_mb = path.stat().st_size / 1_000_000
    print(f"  {level['filename']:<40} {size_mb:.2f} MB")

print()

# Spot-check: load the fine level and verify structure
with open(output_dir / "railroads_fine.geojson") as f:
    check = json.load(f)

print(f"Fine level spot-check:")
print(f"  type:          {check['type']}")
print(f"  feature count: {len(check['features']):,}")
print(f"  first feature geometry type: {check['features'][0]['geometry']['type']}")
print(f"  first feature coord count:   {len(check['features'][0]['geometry']['coordinates'])}")

## Exercise A

The coarse level applies a `scalerank <= 4` filter before simplification. Modify the pipeline (in a copy below) to run the coarse level **without** the scalerank filter.

Compare:
- How many features does the unfiltered coarse level contain?
- How much larger is the file?
- Is the scalerank filter worth keeping, or is geometry simplification alone sufficient?

Write your conclusion as a comment.

In [ ]:
import json
import os
import tempfile

unfiltered = []
for feature in features:
    result = simplify_feature(feature, epsilon=1.0)
    if result is not None:
        unfiltered.append(result)

filtered_path = output_dir / 'railroads_coarse.geojson'
filtered_size = filtered_path.stat().st_size
filtered_count = len(json.load(open(filtered_path))['features'])

fd, temp_path = tempfile.mkstemp(suffix='.geojson')
os.close(fd)
with open(temp_path, 'w') as f:
    json.dump({'type': 'FeatureCollection', 'features': unfiltered}, f)
unfiltered_size = Path(temp_path).stat().st_size
os.remove(temp_path)

print(f'Filtered coarse:   {filtered_count:,} features  |  {filtered_size / 1_000_000:.2f} MB')
print(f'Unfiltered coarse: {len(unfiltered):,} features  |  {unfiltered_size / 1_000_000:.2f} MB')
print()
print('Conclusion: the scalerank filter is worth keeping because simplification alone barely reduces feature count, while the filtered coarse layer is much smaller and faster to render.')

## Exercise B

The pipeline writes GeoJSON using `json.dump()`, which produces unformatted output (no indentation). This is intentional — indentation adds whitespace that inflates file size.

1. Write one of the LOD files again with `indent=2` to make it human-readable.
2. Compare the file size before and after.
3. By what percentage does indentation increase file size?

Do not keep the indented file — overwrite it with the compact version when you are done.

In [ ]:
import json

target_path = output_dir / 'railroads_coarse.geojson'
original_text = target_path.read_text()
original_size = target_path.stat().st_size
data = json.loads(original_text)

with open(target_path, 'w') as f:
    json.dump(data, f, indent=2)
indented_size = target_path.stat().st_size

with open(target_path, 'w') as f:
    json.dump(data, f)
restored_size = target_path.stat().st_size

increase_pct = (indented_size - original_size) / original_size * 100
print(f'Compact size:  {original_size:,} bytes')
print(f'Indented size: {indented_size:,} bytes')
print(f'Increase:      {increase_pct:.1f}%')
print(f'Restored size: {restored_size:,} bytes')

## Check Your Understanding

We use `shapely.simplify(epsilon, preserve_topology=False)`.

Shapely also has `preserve_topology=True`. Look up what that parameter does, then answer:

For a railroad LOD pipeline, should we use `True` or `False`? Why?

Write a 2–3 sentence answer.

For a railroad LOD pipeline, `preserve_topology=False` is usually the better choice because we care most about speed and aggressive vertex reduction on independent line features. `preserve_topology=True` spends extra work preventing topological problems that matter more for polygons and shared boundaries than for standalone railroad segments. If we were simplifying a connected network for routing, that tradeoff might change.

## Next

In [02 — Comparing the Levels](./02-Comparing_the_Levels.ipynb), we load all four output files and compare them visually on a map.